# Chapter 12: Asymptotic Theory of the MLE and the Delta Method

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch12/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_12.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## The Delta Method for the Odds

*Source file:* `code/ch12/c12_1_delta_method_odds.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
p, n = 0.3, 500
samples = rng.binomial(1, p, size=(100_000, n))
tau_hat = np.mean(samples, axis=1) / (1.0 - np.mean(samples, axis=1))
print(f"Delta theoretical variance: {(p/((1-p)**3))/n:.6f} | Empirical: {np.var(tau_hat, ddof=1):.6f}")
eta_hat = np.log(tau_hat)
print(f"Log-odds delta variance: {1/(p*(1-p)*n):.6f} | Empirical: {np.var(eta_hat, ddof=1):.6f}")

Delta theoretical variance: 0.001749 | Empirical: 0.001764
Log-odds delta variance: 0.009524 | Empirical: 0.009570


## Asymptotic Normality of the MLE and Coverage of the Wald Interval

For the Gamma shape parameter $\alpha$ (rate known, $\beta=1$), $I_1(\alpha)=\psi'(\alpha)$ (trigamma). The MLE solves $\psi(\hat\alpha)=\overline{\ln X}$.

*Source file:* `code/ch12/c12_2_mle_asymptotic_normality_wald.py`

In [3]:
import numpy as np
from scipy.special import digamma, polygamma
from scipy import stats

rng = np.random.default_rng(42)
alpha0, R = 2.0, 50_000
def mle_alpha(lx_bar):
    a = np.full_like(lx_bar, 1.0)
    for _ in range(50):                                   # vectorised Newton
        a = np.maximum(a - (digamma(a) - lx_bar) / polygamma(1, a), 1e-6)
    return a
for n in [10, 50, 400]:
    X = rng.gamma(alpha0, 1.0, size=(R, n))
    a_hat = mle_alpha(np.log(X).mean(axis=1))
    z = np.sqrt(n * polygamma(1, alpha0)) * (a_hat - alpha0)
    se = 1 / np.sqrt(n * polygamma(1, a_hat))
    coverage = np.mean(np.abs(a_hat - alpha0) <= 1.96 * se)
    print(f"n={n:4d} | bias={a_hat.mean()-alpha0:+.4f} | n*Var={n*a_hat.var():.4f} (1/I={1/polygamma(1,alpha0):.4f})"
          f" | skewness(z)={stats.skew(z):+.3f} | Wald 95% coverage={coverage:.4f}")

n=  10 | bias=+0.0471 | n*Var=1.6058 (1/I=1.5505) | skewness(z)=+0.486 | Wald 95% coverage=0.9476


n=  50 | bias=+0.0113 | n*Var=1.5671 (1/I=1.5505) | skewness(z)=+0.219 | Wald 95% coverage=0.9501


n= 400 | bias=+0.0012 | n*Var=1.5378 (1/I=1.5505) | skewness(z)=+0.075 | Wald 95% coverage=0.9522


## ARE, Variance Stabilisation and the Second-Order Delta Method

*Source file:* `code/ch12/c12_3_are_variance_stabilisation.py`

In [4]:
import numpy as np

rng = np.random.default_rng(42)
n, R = 101, 100_000
for name, X in [("Normal ", rng.normal(size=(R, n))),
              ("Laplace", rng.laplace(size=(R, n))),
              ("t_3    ", rng.standard_t(3, size=(R, n)))]:
    vm, vmed = X.mean(axis=1).var(), np.median(X, axis=1).var()
    print(f"{name} | n*Var(mean)={n*vm:.3f} | n*Var(median)={n*vmed:.3f} | ARE(median, mean)={vm/vmed:.3f}")

print("Poisson: Var(2*sqrt(Xbar)) * n  (target 1, independent of lambda)")
for lam in [1, 5, 50]:
    Xb = rng.poisson(lam, size=(R, 200)).mean(axis=1)
    print(f"  lambda={lam:3d} | n*Var(Xbar)={200*Xb.var():7.3f} | n*Var(2 sqrt(Xbar))={200*np.var(2*np.sqrt(Xb)):.3f}")

n = 400
pb = rng.binomial(n, 0.5, size=R) / n
W = n * (pb*(1 - pb) - 0.25)
print(f"Second-order delta: E[W]={W.mean():.4f} (theory -0.25), Var[W]={W.var():.4f} (theory 2/16={2/16:.4f})")

Normal  | n*Var(mean)=0.999 | n*Var(median)=1.561 | ARE(median, mean)=0.640


Laplace | n*Var(mean)=2.003 | n*Var(median)=1.156 | ARE(median, mean)=1.733
t_3     | n*Var(mean)=2.980 | n*Var(median)=1.862 | ARE(median, mean)=1.601
Poisson: Var(2*sqrt(Xbar)) * n  (target 1, independent of lambda)


  lambda=  1 | n*Var(Xbar)=  1.004 | n*Var(2 sqrt(Xbar))=1.006


  lambda=  5 | n*Var(Xbar)=  5.022 | n*Var(2 sqrt(Xbar))=1.005


  lambda= 50 | n*Var(Xbar)= 50.119 | n*Var(2 sqrt(Xbar))=1.002
Second-order delta: E[W]=-0.2487 (theory -0.25), Var[W]=0.1237 (theory 2/16=0.1250)
